# Model Performance Data


In [1]:
import json
import platform
import subprocess
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import your
from tqdm.auto import tqdm


## Paths


In [2]:
import runpy

config_path = Path.cwd() / 'config.py'
if not config_path.is_file():
    config_path = Path.cwd().parent / 'config.py'
config = runpy.run_path(str(config_path))
filterbank_file_path = config['filterbank_file_path']
run_dir = config['run_dir']

global_metrics_path = run_dir / 'global_test_segment_metrics.csv'
cnn_checkpoint_path = run_dir / 'best_model_by_acc.pt'
statistical_model_path = run_dir / 'best_statistical_model.joblib'
timings_path = run_dir / 'model_performance_raw_timings.csv'
summary_path = run_dir / 'model_performance_summary.csv'
environment_path = run_dir / 'model_performance_environment.json'
replace_existing_outputs = True

for input_path in (filterbank_file_path, global_metrics_path, cnn_checkpoint_path, statistical_model_path):
    if not input_path.is_file():
        raise FileNotFoundError(f'Input file not found: {input_path}')

for output_path in (timings_path, summary_path, environment_path):
    if output_path.exists() and not replace_existing_outputs:
        raise FileExistsError(f'Performance output already exists: {output_path}')


## Measurement Contract


In [3]:
n_channels = 256
n_time_samples = 256
sample_interval_us = 102.4
n_segments = 1_000
warmup_calls = 20
random_state = 42

segment_budget_ms = n_time_samples * sample_interval_us * 1e-3

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for the CNN GPU measurement')

print(f'Segment budget: {segment_budget_ms:.4f} ms')
print('GPU:', torch.cuda.get_device_name(0))


Segment budget: 26.2144 ms
GPU: Quadro RTX 6000


## Held-Out Segments


In [4]:
global_metrics = pd.read_csv(global_metrics_path)
required_metric_columns = {'model', 'segment_index'}
if not required_metric_columns.issubset(global_metrics.columns):
    raise ValueError('Global-test metrics have an unexpected schema')

cnn_segments = set(global_metrics.loc[global_metrics['model'].eq('CNN'), 'segment_index'].astype(int))
statistical_segments = set(
    global_metrics.loc[global_metrics['model'].str.startswith('Statistical '), 'segment_index'].astype(int)
)
if not cnn_segments or cnn_segments != statistical_segments:
    raise ValueError('Global-test models do not have the same segment set')
if len(cnn_segments) < n_segments:
    raise ValueError('Not enough held-out segments for the measurement')

rng = np.random.default_rng(random_state)
selected_segment_indices = rng.choice(
    np.array(sorted(cnn_segments), dtype=np.int64), size=n_segments, replace=False
)

print('Held-out segments:', len(cnn_segments))
print('Measured segments:', len(selected_segment_indices))


Held-out segments: 14209
Measured segments: 1000


## Models


In [5]:
class CNN1D_RFI_256_logits(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7)
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=7)
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=10)
        self.pool3 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(6400, 256)
        self.fc2 = nn.Linear(256, 1)

    def forward(self, inputs):
        outputs = self.pool1(F.relu(self.conv1(inputs)))
        outputs = F.relu(self.conv2(outputs))
        outputs = F.pad(outputs, (0, 1))
        outputs = self.pool2(outputs)
        outputs = self.pool3(F.relu(self.conv3(outputs)))
        outputs = self.flatten(outputs)
        outputs = self.dropout(F.relu(self.fc1(outputs)))
        return self.fc2(outputs)


cnn_checkpoint = torch.load(cnn_checkpoint_path, map_location='cpu')
required_checkpoint_keys = {
    'epoch', 'model_state_dict', 'optimizer_state_dict',
    'validation_loss', 'validation_accuracy'
}
if set(cnn_checkpoint) != required_checkpoint_keys:
    raise ValueError('CNN checkpoint has an unexpected schema')

cnn_cpu = CNN1D_RFI_256_logits().cpu()
cnn_cpu.load_state_dict(cnn_checkpoint['model_state_dict'])
cnn_cpu.eval()

cnn_gpu = CNN1D_RFI_256_logits().cuda()
cnn_gpu.load_state_dict(cnn_checkpoint['model_state_dict'])
cnn_gpu.eval()

statistical_model = joblib.load(statistical_model_path)
required_bundle_keys = {
    'model_name', 'feature_columns', 'threshold', 'pipeline',
    'validation_metrics', 'test_metrics'
}
if set(statistical_model) != required_bundle_keys:
    raise ValueError('Statistical-model bundle has an unexpected schema')
if len(statistical_model['feature_columns']) != 3:
    raise ValueError('Statistical-model bundle must use three features')

cnn_threshold = 0.5
statistical_threshold = float(statistical_model['threshold'])

print('CNN threshold:', cnn_threshold)
print('Statistical model:', statistical_model['model_name'])
print('Statistical threshold:', statistical_threshold)


CNN threshold: 0.5
Statistical model: MLP
Statistical threshold: 0.5800000000000001


## Prepared Inputs


In [6]:
filterbank_file = your.Your(str(filterbank_file_path))
if filterbank_file.your_header.nchans != n_channels:
    raise ValueError('Filterbank channel count is not 256')


def load_segment(segment_index):
    segment = filterbank_file.get_data(
        nstart=int(segment_index) * n_time_samples,
        nsamp=n_time_samples
    ).T
    if segment.shape != (n_channels, n_time_samples):
        raise ValueError(f'Unexpected segment shape for {segment_index}: {segment.shape}')
    return np.asarray(segment, dtype=np.float32)


def compute_stats(values, axis=None):
    values = np.asarray(values)
    mean = values.mean(axis=axis, keepdims=True)
    std = values.std(axis=axis, keepdims=True)
    centered = values - mean
    std_safe = np.where(std == 0, 1.0, std)
    skew = (centered ** 3).mean(axis=axis, keepdims=True) / (std_safe ** 3)
    kurt = (centered ** 4).mean(axis=axis, keepdims=True) / (std_safe ** 4) - 3.0
    return mean.squeeze(), std.squeeze(), skew.squeeze(), kurt.squeeze()


def normalize_per_channel(segment):
    channel_min = segment.min(axis=1, keepdims=True)
    channel_max = segment.max(axis=1, keepdims=True)
    channel_range = channel_max - channel_min
    channel_range[channel_range == 0] = 1.0
    return 256.0 * (segment - channel_min) / channel_range


def compute_ratio(channel_values, segment_value):
    if segment_value == 0:
        return np.full_like(channel_values, np.nan, dtype=float)
    return np.asarray(channel_values, dtype=float) / segment_value


def segment_features(segment):
    mean_o_segment, std_o_segment, skew_o_segment, kurt_o_segment = compute_stats(segment)
    normalized_segment = normalize_per_channel(segment)
    mean_n_segment, std_n_segment, skew_n_segment, kurt_n_segment = compute_stats(normalized_segment)
    mean_o, std_o, skew_o, kurt_o = compute_stats(segment, axis=1)
    mean_n, std_n, skew_n, kurt_n = compute_stats(normalized_segment, axis=1)
    return pd.DataFrame({
        'mean_o': mean_o, 'std_o': std_o, 'skew_o': skew_o, 'kurt_o': kurt_o,
        'mean_n': mean_n, 'std_n': std_n, 'skew_n': skew_n, 'kurt_n': kurt_n,
        'mean_o_ratio': compute_ratio(mean_o, mean_o_segment),
        'std_o_ratio': compute_ratio(std_o, std_o_segment),
        'skew_o_ratio': compute_ratio(skew_o, skew_o_segment),
        'kurt_o_ratio': compute_ratio(kurt_o, kurt_o_segment),
        'mean_n_ratio': compute_ratio(mean_n, mean_n_segment),
        'std_n_ratio': compute_ratio(std_n, std_n_segment),
        'skew_n_ratio': compute_ratio(skew_n, skew_n_segment),
        'kurt_n_ratio': compute_ratio(kurt_n, kurt_n_segment)
    })


segments = np.empty((n_segments, n_channels, n_time_samples), dtype=np.float32)
for order, segment_index in enumerate(tqdm(selected_segment_indices, desc='Loading segments')):
    segments[order] = load_segment(segment_index)

channel_maxima = np.max(np.abs(segments), axis=2, keepdims=True)
channel_maxima[channel_maxima < 1e-8] = 1.0
cnn_cpu_inputs = torch.from_numpy(segments / channel_maxima).unsqueeze(2).contiguous()
cnn_gpu_inputs = cnn_cpu_inputs.to('cuda')

statistical_inputs = []
for segment in tqdm(segments, desc='Calculating features'):
    features = segment_features(segment).loc[:, statistical_model['feature_columns']]
    if not np.isfinite(features.to_numpy()).all():
        raise ValueError('Statistical-model input contains non-finite values')
    statistical_inputs.append(features)

if cnn_cpu_inputs.shape != (n_segments, n_channels, 1, n_time_samples):
    raise ValueError('CNN input has an unexpected shape')
if len(statistical_inputs) != n_segments:
    raise ValueError('Statistical input count is invalid')

del segments
print('CNN input shape:', tuple(cnn_cpu_inputs.shape))
print('Statistical inputs:', len(statistical_inputs))


Loading segments:   0%|          | 0/1000 [00:00<?, ?it/s]

Calculating features:   0%|          | 0/1000 [00:00<?, ?it/s]

CNN input shape: (1000, 256, 1, 256)
Statistical inputs: 1000


## Inference Timing


In [7]:
def statistical_scores(features):
    pipeline = statistical_model['pipeline']
    if hasattr(pipeline, 'predict_proba'):
        return pipeline.predict_proba(features)[:, 1]
    return 1.0 / (1.0 + np.exp(-pipeline.decision_function(features)))


@torch.inference_mode()
def benchmark_cnn(model, inputs, device_name):
    for input_tensor in inputs[:warmup_calls]:
        _ = torch.sigmoid(model(input_tensor)) >= cnn_threshold
    if device_name == 'cuda':
        torch.cuda.synchronize()

    rows = []
    for order, input_tensor in enumerate(tqdm(inputs, desc=f'CNN {device_name}')):
        if device_name == 'cuda':
            torch.cuda.synchronize()
        started = time.perf_counter()
        _ = torch.sigmoid(model(input_tensor)) >= cnn_threshold
        if device_name == 'cuda':
            torch.cuda.synchronize()
        rows.append({
            'model': 'CNN',
            'device': device_name,
            'segment_order': order,
            'segment_index': int(selected_segment_indices[order]),
            'duration_ms': (time.perf_counter() - started) * 1e3
        })
    return rows


def benchmark_statistical():
    for features in statistical_inputs[:warmup_calls]:
        _ = statistical_scores(features) >= statistical_threshold

    rows = []
    for order, features in enumerate(tqdm(statistical_inputs, desc='Statistical CPU')):
        started = time.perf_counter()
        _ = statistical_scores(features) >= statistical_threshold
        rows.append({
            'model': f"Statistical {statistical_model['model_name']}",
            'device': 'cpu',
            'segment_order': order,
            'segment_index': int(selected_segment_indices[order]),
            'duration_ms': (time.perf_counter() - started) * 1e3
        })
    return rows


timing_rows = []
timing_rows.extend(benchmark_cnn(cnn_cpu, cnn_cpu_inputs, 'cpu'))
timing_rows.extend(benchmark_cnn(cnn_gpu, cnn_gpu_inputs, 'cuda'))
timing_rows.extend(benchmark_statistical())

timings = pd.DataFrame(timing_rows)
if len(timings) != 3 * n_segments:
    raise RuntimeError('Timing row count is invalid')
timings['n_segments'] = n_segments
timings['warmup_calls'] = warmup_calls
timings['segment_budget_ms'] = segment_budget_ms
timings['timing_scope'] = 'model_and_threshold_only'
timings.to_csv(timings_path, index=False)


CNN cpu:   0%|          | 0/1000 [00:00<?, ?it/s]

CNN cuda:   0%|          | 0/1000 [00:00<?, ?it/s]

Statistical CPU:   0%|          | 0/1000 [00:00<?, ?it/s]

## Saved Results


In [8]:
summary = (
    timings.groupby(['model', 'device'], as_index=False)
    .agg(
        n_segments=('duration_ms', 'size'),
        median_ms=('duration_ms', 'median'),
        mean_ms=('duration_ms', 'mean'),
        p95_ms=('duration_ms', lambda values: np.percentile(values, 95)),
        min_ms=('duration_ms', 'min'),
        max_ms=('duration_ms', 'max')
    )
)
summary['segment_budget_ms'] = segment_budget_ms
summary['meets_budget_at_p95'] = summary['p95_ms'] <= segment_budget_ms
summary['budget_factor_at_p95'] = segment_budget_ms / summary['p95_ms']
summary['timing_scope'] = 'model_and_threshold_only'
summary.to_csv(summary_path, index=False)


def git_revision():
    try:
        return subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'],
            text=True,
            stderr=subprocess.DEVNULL
        ).strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None


environment = {
    'code_revision': git_revision(),
    'python_version': platform.python_version(),
    'torch_version': torch.__version__,
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
    'cuda_available': True,
    'cuda_device': torch.cuda.get_device_name(0),
    'n_segments': n_segments,
    'warmup_calls': warmup_calls,
    'random_state': random_state,
    'segment_definition': '256 channels by 256 time samples',
    'segment_budget_ms': segment_budget_ms,
    'timing_scope': (
        'model inference and threshold decision only; excludes file reading, '
        'feature calculation, CNN normalization, and CPU-to-GPU transfer'
    )
}
with environment_path.open('w', encoding='utf-8') as handle:
    json.dump(environment, handle, indent=2, sort_keys=True)
    handle.write('\n')

print('Raw timings:', timings_path)
print('Summary:', summary_path)
print('Environment:', environment_path)
display(summary.sort_values(['model', 'device']).reset_index(drop=True))


Raw timings: /u/akazantsev/ml_dl_for_rfi_mitigation/runs/punch_rfi_reproduction/model_performance_raw_timings.csv
Summary: /u/akazantsev/ml_dl_for_rfi_mitigation/runs/punch_rfi_reproduction/model_performance_summary.csv
Environment: /u/akazantsev/ml_dl_for_rfi_mitigation/runs/punch_rfi_reproduction/model_performance_environment.json


,model,device,n_segments,median_ms,mean_ms,p95_ms,min_ms,max_ms,segment_budget_ms,meets_budget_at_p95,budget_factor_at_p95,timing_scope
0,CNN,cpu,1000,89.838798,90.139782,91.590059,88.211934,105.272187,26.2144,False,0.286214,model_and_threshold_only
1,CNN,cuda,1000,2.076218,2.101028,2.242722,2.053959,7.121763,26.2144,True,11.688653,model_and_threshold_only
2,Statistical MLP,cpu,1000,1.322644,1.328660,1.377310,1.301912,1.480864,26.2144,True,19.033040,model_and_threshold_only
